# Notebook 2: py-SpaceTrooper Tutorial — CosMx 数据

本教程演示如何使用 py-SpaceTrooper 对 CosMx 空间转录组数据进行质量控制。

## 管道概览
1. 加载数据
2. 计算每细胞 QC 指标
3. 检测异常值
4. 计算 QC 评分
5. 可视化结果

In [ ]:
import numpy as np
import anndata as ad
import matplotlib.pyplot as plt

# 方法 1: 使用函数式 API
from spacetrooper import (
    spatial_per_cell_qc,
    compute_outliers_qc_score,
    check_outliers,
    compute_qc_score,
    compute_threshold_flags,
    plot_metric_hist,
    plot_centroids,
)

In [ ]:
# 加载 CosMx 测试数据
adata = ad.read_h5ad('../data/fixture_cosmx.h5ad')
print(f'Loaded: {adata.n_obs} cells, {adata.n_vars} genes')
print(f'Technology: {adata.uns["technology"]}')

## Step 1: 计算每细胞 QC 指标

`spatial_per_cell_qc` 计算以下指标并添加到 `adata.obs`:
- `sum` / `detected`: 总转录本数 / 检测到的基因数
- `control_sum` / `target_sum`: 阴性探针 / 目标基因转录本数
- `ctrl_total_ratio`: 阴性探针占比
- `log2SignalDensity`: log2(转录本密度)
- `Area_um`: 细胞面积 (μm²)
- `log2AspectRatio`: log2(长宽比)

In [ ]:
np.random.seed(42)
spatial_per_cell_qc(adata)

print(f'After QC: {adata.n_obs} cells')
print(f'QC columns: {[c for c in adata.obs.columns if "log2" in c or "ctrl" in c or "target" in c]}')

## Step 2: 检测异常值

`compute_outliers_qc_score` 对每个 QC 指标检测异常值:
- 使用 medcouple (偏斜分布) 或 MAD (对称分布)
- 自动生成训练集用于 QC 评分

In [ ]:
compute_outliers_qc_score(adata)
check_outliers(adata, verbose=True)

## Step 3: 计算 QC 评分

`compute_qc_score` 使用岭逻辑回归计算每细胞的 QC 评分 (0-1):
- 训练集: 平衡的好/坏细胞
- 特征: QC 指标 + 交互项
- 正则化: L2 (ridge)

In [ ]:
compute_qc_score(adata, verbose=False)

print(f'QC_score: mean={adata.obs["QC_score"].mean():.4f}, '
      f'min={adata.obs["QC_score"].min():.4f}, '
      f'max={adata.obs["QC_score"].max():.4f}')

## Step 4: 可视化

In [ ]:
# QC 指标直方图
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

plot_metric_hist(adata, 'log2SignalDensity', ax=axes[0, 0])
plot_metric_hist(adata, 'Area_um', ax=axes[0, 1])
plot_metric_hist(adata, 'log2AspectRatio', ax=axes[1, 0])
plot_metric_hist(adata, 'log2Ctrl_total_ratio', ax=axes[1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# QC 评分空间分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_centroids(adata, colour_by='QC_score', ax=axes[0])
axes[0].set_title('QC Score')

plot_centroids(adata, colour_by='log2SignalDensity', ax=axes[1])
axes[1].set_title('log2 Signal Density')

plt.tight_layout()
plt.show()

## 方法 2: 使用类 API

SpaceTrooper 类提供 method chaining API:

In [ ]:
from spacetrooper import SpaceTrooper

np.random.seed(42)
st = SpaceTrooper(adata.copy())
st.spatial_per_cell_qc()
st.compute_qc_score()

print(st)
print(f'QC_score mean: {st.adata.obs["QC_score"].mean():.4f}')